# Fraud Detection with CatBoost

This notebook builds a leakage-aware fraud classifier, tunes CatBoost with stratified cross-validation, selects a decision threshold on a validation set, and reports final performance on a held-out test set. Average precision is the primary metric because fraud cases are rare.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

RANDOM_STATE = 42


## 1. Load and inspect the data

In [ ]:
# Load the feature-engineered transaction dataset.
data_path = Path("../Datasets/Transaction_risk_insights_feature_engineered.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


## 2. Select and prepare model features

In [ ]:
# Use transaction-time attributes only; exclude identifiers and the target.
features = [
    "account_type",
    "transaction_type",
    "transaction_amount",
    "transaction_direction",
    "account_balance",
    "merchant_category",
    "state",
    "credit_score",
    "has_loan",
    "loan_type",
    "emi_amount",
    "channel",
    "kyc_status",
    "transaction_hour",
    "transaction_month",
    "day_of_week",
    "is_weekend",
]


In [ ]:
target = "is_fraud"
missing_columns = sorted(set(features + [target]) - set(df.columns))
if missing_columns:
    raise KeyError(f"Missing required dataset columns: {missing_columns}")

X = df.loc[:, features].copy()
y = df[target].astype("int8").copy()
print(f"Feature matrix: {X.shape}; target distribution: {y.value_counts().sort_index().to_dict()}")


In [ ]:
# Log-transform skewed non-negative monetary features.
for column in ("transaction_amount", "account_balance", "emi_amount"):
    if (X[column] < 0).any():
        raise ValueError(f"{column} contains negative values; log1p is not valid.")
    X[f"{column}_log"] = np.log1p(X[column])

X = X.drop(columns=["transaction_amount", "account_balance", "emi_amount"])
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Prepared features: {X.shape[1]} total; {len(categorical_cols)} categorical")
print("Categorical columns:", categorical_cols)


## 3. Create stratified train, validation, and test splits

In [ ]:
# Reserve a stratified test set, then split training data for tuning and threshold selection.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_tune_train, X_valid, y_tune_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

print(f"Tuning/training rows: {len(X_tune_train):,}")
print(f"Threshold-validation rows: {len(X_valid):,}")
print(f"Held-out test rows: {len(X_test):,}")


In [ ]:
# Compute imbalance weights from the model-training partition only.
negative_count = int((y_tune_train == 0).sum())
positive_count = int((y_tune_train == 1).sum())
if positive_count == 0:
    raise ValueError("The model-training partition contains no fraud examples.")

# Python floats keep estimator parameters compatible with cloning and serialization.
class_weights = (1.0, float(negative_count / positive_count))
print(f"Non-fraud: {negative_count:,}; fraud: {positive_count:,}")
print(f"Class weights: {class_weights}")


## 4. Tune and train CatBoost

In [ ]:
# Search a compact range suited to imbalanced tabular classification.
catboost_param_distributions = {
    "iterations": randint(150, 401),
    "learning_rate": uniform(0.02, 0.13),
    "depth": randint(4, 9),
    "l2_leaf_reg": uniform(2.0, 8.0),
    "random_strength": uniform(0.1, 2.9),
    "bagging_temperature": uniform(0.0, 3.0),
}


In [ ]:
catboost_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        class_weights=class_weights,
        random_seed=RANDOM_STATE,
        verbose=False,
        thread_count=1,
        allow_writing_files=False,
    ),
    param_distributions=catboost_param_distributions,
    n_iter=6,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)


In [ ]:
# Tune against the same average-precision metric used for model evaluation.
# Use a stratified sample for search, then refit the winner on all tuning rows.
search_size = min(80_000, len(X_tune_train))
if search_size < len(X_tune_train):
    X_search, _, y_search, _ = train_test_split(
        X_tune_train,
        y_tune_train,
        train_size=search_size,
        random_state=RANDOM_STATE,
        stratify=y_tune_train,
    )
else:
    X_search = X_tune_train
    y_search = y_tune_train
catboost_search.fit(
    X_search,
    y_search,
    cat_features=categorical_cols,
)
best_params = catboost_search.best_params_
print(f"Best 3-fold CV average precision: {catboost_search.best_score_:.4f}")
print("Best CatBoost parameters:", best_params)

catboost_model = CatBoostClassifier(
    **best_params,
    loss_function="Logloss",
    eval_metric="PRAUC",
    class_weights=class_weights,
    random_seed=RANDOM_STATE,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)
catboost_model.fit(
    X_tune_train,
    y_tune_train,
    cat_features=categorical_cols,
)


## 5. Select a threshold and evaluate the final model

In [ ]:
# Select the operating threshold on validation data, never on the held-out test set.
valid_probabilities = catboost_model.predict_proba(X_valid)[:, 1]
valid_precision, valid_recall, thresholds = precision_recall_curve(
    y_valid,
    valid_probabilities,
)
valid_f1 = (
    2 * valid_precision[:-1] * valid_recall[:-1]
    / (valid_precision[:-1] + valid_recall[:-1] + 1e-12)
)
best_threshold_index = int(np.argmax(valid_f1))
decision_threshold = float(thresholds[best_threshold_index])
print(f"Validation-selected threshold (max F1): {decision_threshold:.4f}")
print(f"Validation F1 at selected threshold: {valid_f1[best_threshold_index]:.4f}")

# Evaluate the selected model and threshold once on the untouched test set.
test_probabilities = catboost_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= decision_threshold).astype("int8")
test_average_precision = average_precision_score(y_test, test_probabilities)
test_baseline_ap = float(y_test.mean())
print(f"\nTest average precision: {test_average_precision:.4f}")
print(f"No-skill baseline (fraud prevalence): {test_baseline_ap:.4f}")
print(f"Average-precision lift over baseline: {test_average_precision / test_baseline_ap:.2f}x")
print("\nTest classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Non-Fraud", "Fraud"],
        zero_division=0,
    )
)

matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    matrix,
    annot=True,
    fmt=",d",
    cmap="Blues",
    xticklabels=["Non-Fraud", "Fraud"],
    yticklabels=["Non-Fraud", "Fraud"],
    ax=ax,
)
ax.set(title="CatBoost - Held-out Test Confusion Matrix", xlabel="Predicted", ylabel="Actual")
fig.tight_layout()
plt.show()
